# 05 Outfit Completion and Responsible Recommendation
Week 5 deliverables: the outfit-aware feature the project is named for, and the evidence for the responsible-recommendation section of the report.

**Part A. Outfit completion ("complete the look").** Given one item, suggest items from *other* product types that customers bought on the same day. Notebook 04 showed that a product-type affinity re-rank adds almost nothing to the personalised lists, so outfit-awareness is evaluated here on the task it actually fits: completing a basket from one anchor item.

**Part B. Cleaner lists.** Collapse colour variants of one product in the final lists and measure what it costs.

**Part C. Responsible recommendation.** Popularity bias and exposure, performance across age groups, filter bubbles, cold start and privacy, using the final test-time model.

Run notebooks 02, 03 and 04 first (04 saves the final model).

In [1]:
import json, pickle, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse

from src import config, data_audit as da, prepare as prep, evaluation as ev, recommender as rec, hybrid as hy, outfit as of

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(config.IMAGES_DIR / name, dpi=150)
    plt.show()

## 1. Load data and the final model

In [2]:
articles = da.load_articles()
articles_by_id = articles.set_index("article_id")
prepared = prep.load_prepared()
exp = prep.build_experiment(prepared, articles)
with open(config.CLEANED_DIR / "final_hybrid.pkl", "rb") as f:
    bundle = pickle.load(f)
final, settings = bundle["model"], bundle["settings"]
assert final.items.equals(exp.items), "final model and prepared data do not match; re-run notebooks 02 and 04"

K = config.TOP_K
cleaned = prepared.cleaned
val_start, test_start = pd.Timestamp(prepared.meta["val_start"]), pd.Timestamp(prepared.meta["test_start"])
train_df, val_df, test_df = exp.split.train, exp.split.val, exp.split.test
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
names, types = exp.item_names, exp.category_names[exp.category_of_item]
print(f"universe: {len(exp.users):,} users, {len(exp.items):,} items | test starts {test_start.date()}")

universe: 45,078 users, 11,916 items | test starts 2020-09-09


## Part A. Outfit completion
**Data.** A *basket* is everything one customer bought on one day (2 to 8 items). **Model.** Two items are similar if they appear in the same baskets (cosine across baskets). **Task.** For a held-out basket, show one item (the *anchor*) and ask the model to recover the *other items of a different product type* from the basket. A complement is never the same type or product as the anchor.

**Baselines.** Trending items (different type), content similarity (similar style, different type) and random items (different type).

In [3]:
pop_ranking = final.pop_ranking
train_baskets = of.basket_matrix(train_df, exp.users, exp.items)
val_baskets = of.basket_matrix(val_df, exp.users, exp.items)
print(f"training baskets: {train_baskets.shape[0]:,} | validation baskets: {val_baskets.shape[0]:,}")

def outfit_metrics(recs, truth):
    per = ev.precision_recall_ndcg(recs, truth)
    return {"Hit@10": (per["precision"] > 0).mean(), "Recall@10": per["recall"].mean(), "NDCG@10": per["ndcg"].mean(),
            "Coverage": ev.catalog_coverage(recs, len(exp.items))}

anchors_val, truth_val = of.anchor_targets(val_baskets, exp.category_of_item, config.OUTFIT_EVAL_PAIRS, config.RANDOM_SEED)
rows = []
for shrink in config.OUTFIT_SHRINK:
    table = of.fit_outfit_table(train_baskets, shrink=shrink)
    recs = of.complement_candidates(table, anchors_val, exp.category_of_item, exp.product_of_item, K, pop_ranking)
    rows.append({"shrink": shrink, **outfit_metrics(recs, truth_val)})
outfit_grid = pd.DataFrame(rows).sort_values("Recall@10", ascending=False).reset_index(drop=True)
display(outfit_grid)
best_shrink = float(outfit_grid.iloc[0]["shrink"])
print(f"{len(anchors_val):,} validation (basket, anchor) pairs | selected shrink: {best_shrink}")

training baskets: 160,844 | validation baskets: 3,925


,shrink,Hit@10,Recall@10,NDCG@10,Coverage
0,20.0000,0.0906,0.0540,0.0436,0.7908
1,5.0000,0.0874,0.0520,0.0422,0.7686
2,0.0000,0.0855,0.0510,0.0414,0.7545


9,963 validation (basket, anchor) pairs | selected shrink: 20.0


### Final outfit evaluation on the test period
The co-purchase model is refitted on training plus validation baskets and evaluated once on test baskets.

In [4]:
test_baskets = of.basket_matrix(test_df, exp.users, exp.items)
final_baskets = of.basket_matrix(trainval_df, exp.users, exp.items)
outfit_table = of.fit_outfit_table(final_baskets, shrink=best_shrink)
anchors_test, truth_test = of.anchor_targets(test_baskets, exp.category_of_item, config.OUTFIT_EVAL_PAIRS, config.RANDOM_SEED)
print(f"{test_baskets.shape[0]:,} test baskets -> {len(anchors_test):,} (basket, anchor) pairs")

rng = np.random.default_rng(config.RANDOM_SEED)
def random_complements(anchors):
    out = np.empty((len(anchors), K), dtype=np.int64)
    for r, a in enumerate(anchors):
        while True:
            pick = rng.choice(len(exp.items), 3 * K, replace=False)
            pick = pick[(exp.category_of_item[pick] != exp.category_of_item[a]) & (exp.product_of_item[pick] != exp.product_of_item[a])]
            if len(pick) >= K:
                out[r] = pick[:K]; break
    return out

empty_table = rec.NeighbourTable(np.zeros((len(exp.items), 1), dtype=np.int32), np.zeros((len(exp.items), 1), dtype=np.float32))
candidates = {
    "co-purchase (outfit model)": of.complement_candidates(outfit_table, anchors_test, exp.category_of_item, exp.product_of_item, K, pop_ranking),
    "trending (different type)": of.complement_candidates(empty_table, anchors_test, exp.category_of_item, exp.product_of_item, K, pop_ranking),
    "content similarity (different type)": of.complement_candidates(final._content_table, anchors_test, exp.category_of_item, exp.product_of_item, K, pop_ranking),
    "random (different type)": random_complements(anchors_test),
}
outfit_results = pd.DataFrame({n: outfit_metrics(r, truth_test) for n, r in candidates.items()}).T
display(outfit_results)

hit = lambda recs: (ev.precision_recall_ndcg(recs, truth_test)["precision"] > 0).to_numpy().astype(float)
boot = [{"outfit model minus": n, **ev.paired_bootstrap(hit(candidates["co-purchase (outfit model)"]), hit(r), config.BOOTSTRAP_SAMPLES, config.RANDOM_SEED)}
        for n, r in candidates.items() if n != "co-purchase (outfit model)"]
display(pd.DataFrame(boot))

2,838 test baskets -> 6,833 (basket, anchor) pairs


,Hit@10,Recall@10,NDCG@10,Coverage
co-purchase (outfit model),0.0842,0.0535,0.0428,0.7310
trending (different type),0.0782,0.0392,0.0209,0.0013
content similarity (different type),0.0524,0.0293,0.0162,0.4637
random (different type),0.0012,0.0004,0.0003,0.9969


,outfit model minus,mean_difference,ci_low,ci_high,significant
0,trending (different type),0.0060,-0.0029,0.0148,False
1,content similarity (different type),0.0318,0.0246,0.0394,True
2,random (different type),0.0830,0.0767,0.0902,True


**Reading this.** Hit@10 is the share of (basket, anchor) pairs where at least one other-type item from the basket is in the top 10. Compare the co-purchase model with the trending baseline: if the gap is large, the data do contain a usable "bought together" signal that is *not* just popularity. Content similarity usually does badly here by design, because similar items are substitutes, not complements.

### Examples

In [5]:
rng = np.random.default_rng(config.RANDOM_SEED)
for r in rng.choice(len(anchors_test), 5, replace=False):
    a = anchors_test[r]
    print("=" * 90)
    print(f"Anchor: {names[a]} ({types[a]})")
    for item in candidates["co-purchase (outfit model)"][r][:6]:
        hit_mark = "  <-- in the real basket" if truth_test[r, item] > 0 else ""
        print(f"   + {names[item]} ({types[item]}){hit_mark}")

Anchor: Jackie brief 2pk HR LS (Underwear bottom)
   + DIV Livia tee (T-shirt)
   + Evelina Padded Soft Top (Bra)
   + Honey Leggings 2PACK - TVP (Leggings/Tights)
   + CNY Emma blazer (Blazer)
   + Pen skirt (Skirt)
   + Santorini playsuit (Shorts)
Anchor: Sky wide trouser(1) (Trousers)
   + Shakira pile (Jacket)
   + Taylor top (Top)
   + Osman crew (Sweater)
   + Lucien CONSCIOUS (Blouse)
   + Hilton skirt (Skirt)
   + Morris (Sweater)
Anchor: Timeless HR Cheeky Hipster (Swimwear bottom)
   + Timeless Push Triangle(1) (Bikini top)
   + Lyon Long Top (Bikini top)
   + Tropic Like Its Hot Top (Bikini top)
   + All That Jazz Push Up Bra (Bikini top)
   + Push it Push Bra (Bikini top)
   + Wowcha Top (Bikini top)
Anchor: MINT TANK (Vest top)
   + Elly Fancy (T-shirt)
   + Nora T-shirt (T-shirt)
   + First Night Playsuit (Shorts)
   + Frenchie F&F dress (Dress)
   + X-Tina Long Top (Bikini top)
   + Pretty Tee (T-shirt)
Anchor: Rachel or Merry HW Denim TRS (Trousers)
   + Kiev super tras

In [6]:
with open(config.CLEANED_DIR / "outfit_model.pkl", "wb") as f:
    pickle.dump({"table": outfit_table, "shrink": best_shrink, "basket_sizes": config.OUTFIT_BASKET_SIZES}, f)
print("Saved data/cleaned/outfit_model.pkl")

Saved data/cleaned/outfit_model.pkl


## Part B. Collapse colour variants in the final lists
Week 3 showed that several articles of one product can fill a list (the same design in different colours). Here the final hybrid generates 50 candidates and keeps at most one article per product.

In [7]:
test_users = prep.users_with_interactions(exp.test_matrix)
seen = final.seen
test_new = (exp.test_matrix - exp.test_matrix.multiply(seen)).tocsr(); test_new.eliminate_zeros()
test_users_new = prep.users_with_interactions(test_new)
share_final = seen.getnnz(axis=0) / seen.shape[0]
N_ITEMS = len(exp.items)

ranks_all, ranks_new = final.component_ranks(test_users), final.component_ranks(test_users_new)
w_all, w_new = settings["weights_all_purchases"], settings["weights_discovery"]
g_all, g_new = settings["outfit_gamma_all_purchases"], settings["outfit_gamma_discovery"]

def lists(ranks, users, weights, gamma):
    plain = final.recommend(ranks, weights, users, K, gamma=gamma)
    wide = final.recommend(ranks, weights, users, config.N_CANDIDATES, gamma=gamma)
    return plain, rec.diversify_by_product(wide, exp.product_of_item, K)

plain_all, dedup_all = lists(ranks_all, test_users, w_all, g_all)
plain_new, dedup_new = lists(ranks_new, test_users_new, w_new, g_new)
ev_kw = lambda truth, users: dict(truth=truth, user_positions=users, item_user_share=share_final, n_items=N_ITEMS, category_of_item=exp.category_of_item)
variant_check = pd.DataFrame({
    "all purchases, as in notebook 04": ev.evaluate_topk(plain_all, **ev_kw(exp.test_matrix, test_users)),
    "all purchases, variants collapsed": ev.evaluate_topk(dedup_all, **ev_kw(exp.test_matrix, test_users)),
    "discovery, as in notebook 04": ev.evaluate_topk(plain_new, **ev_kw(test_new, test_users_new)),
    "discovery, variants collapsed": ev.evaluate_topk(dedup_new, **ev_kw(test_new, test_users_new)),
}).T
display(variant_check)
for name, lst in [("all purchases", plain_all), ("discovery", plain_new)]:
    print(f"{name}: average distinct products per list before collapsing = "
          f"{np.mean([len(set(exp.product_of_item[row])) for row in lst]):.2f} of {K}")

,Precision@10,Recall@10,NDCG@10,Coverage,Novelty (bits),Mean popularity,Category diversity,users evaluated
"all purchases, as in notebook 04",0.0109,0.0586,0.0396,0.7807,8.6781,0.0041,0.6190,"5,385.0000"
"all purchases, variants collapsed",0.0093,0.0499,0.0351,0.7949,8.7049,0.0040,0.6819,"5,385.0000"
"discovery, as in notebook 04",0.0088,0.0458,0.0269,0.2550,7.7430,0.0080,0.5632,"5,176.0000"
"discovery, variants collapsed",0.0082,0.0431,0.0255,0.2252,7.7824,0.0078,0.6334,"5,176.0000"


all purchases: average distinct products per list before collapsing = 8.78 of 10
discovery: average distinct products per list before collapsing = 8.44 of 10


Collapsing variants may cost a little accuracy (a customer may well buy another colour), and it gives each list more variety. Decide which the product should favour and say why; the prototype uses the collapsed lists.

## Part C. Responsible recommendation
Each analysis below uses the discovery lists (items the customer has not bought) unless stated. Popularity is the training-period share of customers who bought an item.

### C1. Popularity bias and exposure

In [8]:
train_pop = seen.getnnz(axis=0)
top1 = np.zeros(N_ITEMS, dtype=bool); top1[np.argsort(-train_pop)[: max(1, N_ITEMS // 100)]] = True
pop_lists = prep.recommend_popular(final.pop_scores, seen, test_users_new, K)
knn_lists = final.knn.recommend(final.knn_history, test_users_new, K, final.pop_ranking)
exposure = {"popularity (3-day half-life)": pop_lists, "item-item CF": knn_lists, "hybrid (discovery)": dedup_new}
rows = []
for name, lst in exposure.items():
    counts = np.bincount(lst.ravel(), minlength=N_ITEMS)
    rows.append({"model": name, "share of slots on top-1% items": top1[lst].mean(),
                 "distinct items shown": int((counts > 0).sum()), "catalogue coverage": (counts > 0).mean(),
                 "Gini of exposure": da.gini(pd.Series(counts)), "mean popularity of shown items": share_final[lst].mean()})
exposure_table = pd.DataFrame(rows).set_index("model")
display(exposure_table)

,share of slots on top-1% items,distinct items shown,catalogue coverage,Gini of exposure,mean popularity of shown items
model,,,,,
popularity (3-day half-life),0.2913,14,0.0012,0.9992,0.0080
item-item CF,0.2702,5903,0.4954,0.8524,0.0061
hybrid (discovery),0.2949,2683,0.2252,0.9819,0.0078


A high Gini and a large share of slots on the top 1% of items mean the system keeps showing what is already popular, which concentrates sales and starves new or niche items (a feedback loop if the system's output also drives future purchases).

### C2. Performance across age groups
Age is not a model input. This checks whether the *results* differ by group, using the age recorded in the customer table.

In [9]:
customers = da.load_customers()
age = customers.set_index("customer_id")["age"].reindex(exp.users).to_numpy()
group_of = lambda users: pd.cut(pd.Series(age[users]), [0, 24, 34, 49, 200], labels=["under 25", "25-34", "35-49", "50+"]).astype(object).fillna("age unknown")

def by_group(recs, truth, users):
    per = ev.precision_recall_ndcg(recs, truth[users])["ndcg"].reset_index(drop=True)
    return per.groupby(group_of(users).reset_index(drop=True)).mean()

baseline_discovery = by_group(pop_lists, test_new, test_users_new)
hybrid_discovery = by_group(dedup_new, test_new, test_users_new)
fair = pd.DataFrame({"popularity": baseline_discovery, "hybrid": hybrid_discovery})
fair["hybrid / popularity"] = fair["hybrid"] / fair["popularity"]
fair["customers"] = group_of(test_users_new).value_counts().reindex(fair.index)
display(fair)
known = fair.drop(index="age unknown", errors="ignore")
print(f"Best-to-worst group ratio of hybrid NDCG (known ages): {known['hybrid'].max() / known['hybrid'].min():.2f}")
print(f"Lowest hybrid/popularity ratio across groups: {known['hybrid / popularity'].min():.2f}")

,popularity,hybrid,hybrid / popularity,customers
25-34,0.0219,0.0260,1.1912,1767
35-49,0.0155,0.0237,1.5305,995
50+,0.0212,0.0270,1.2692,1041
age unknown,0.0000,0.0140,inf,13
under 25,0.0208,0.0250,1.2043,1360


Best-to-worst group ratio of hybrid NDCG (known ages): 1.14
Lowest hybrid/popularity ratio across groups: 1.19


A ratio near 1 between the best and worst group, and a hybrid-to-popularity ratio above 1 in every group, would indicate that the personalisation does not help one age group at the expense of another. A group with few customers has a wide uncertainty interval; read small differences cautiously.

### C3. Filter bubbles and variety

In [10]:
def never_bought_type_share(lst, users):
    out = []
    for row, u in zip(lst, users):
        bought_types = set(exp.category_of_item[seen.indices[seen.indptr[u]:seen.indptr[u + 1]]])
        out.append(np.mean([exp.category_of_item[i] not in bought_types for i in row]))
    return float(np.mean(out))

bubble = pd.DataFrame({
    name: {"share of items in a product type never bought": never_bought_type_share(lst, test_users_new),
           "distinct product types per list": float(np.mean([len(set(exp.category_of_item[r])) for r in lst])),
           "distinct products per list": float(np.mean([len(set(exp.product_of_item[r])) for r in lst]))}
    for name, lst in {"popularity": pop_lists, "item-item CF": knn_lists, "hybrid (discovery, variants collapsed)": dedup_new}.items()}).T
display(bubble)
repurchase_share = np.mean([seen[u, i] > 0 for row, u in zip(dedup_all, test_users) for i in row])
print(f"All-purchases hybrid: {repurchase_share:.1%} of recommended slots are items the customer bought before")

,share of items in a product type never bought,distinct product types per list,distinct products per list
popularity,0.4304,5.9847,9.0301
item-item CF,0.1395,4.1820,7.4950
"hybrid (discovery, variants collapsed)",0.3651,6.3337,10.0000


All-purchases hybrid: 74.2% of recommended slots are items the customer bought before


A low share of never-bought product types means lists keep customers inside the categories they already buy from. The all-purchases hybrid is dominated by repurchases, so the prototype shows "Buy again" and "Discover" separately rather than mixing them.

### C4. Cold start

In [11]:
cold = prepared.cold_items
cold_test = cold[cold["t_dat"] >= test_start]
share = len(cold_test) / (len(cold_test) + exp.test_matrix.nnz)
print(f"Test period: {share:.1%} of purchases by evaluated customers are of items outside the model's item universe "
      f"({cold_test['article_id'].nunique():,} distinct items).")
print("Customers without enough training history are excluded from the evaluation and would receive popular items in practice.")
display(prepared.report.loc[[i for i in prepared.report.index if "universe" in i or "cleaned" in i or "cold" in i]])

Test period: 58.9% of purchases by evaluated customers are of items outside the model's item universe (4,717 distinct items).
Customers without enough training history are excluded from the evaluation and would receive popular items in practice.


""


This is the largest limitation of the collaborative components: most purchases in the test period are of items that have too little (or no) purchase history, so only content-based scoring can reach them (notebook 04 shows content similarity at several times the random rate for these items).

### C5. Privacy, manipulation and explainability (no computation)
- **Privacy / data minimisation.** The models use purchase history and product metadata only. Age, postal code and membership fields are not model inputs; age is used here solely to audit fairness. Customer identifiers in the data are hashed, the prototype shows shortened identifiers, and any real deployment would need consent, retention limits and the right to opt out of personalisation.
- **Manipulation and feedback loops.** Popularity-weighted systems can be gamed by bulk purchases and reinforce themselves when recommendations drive the purchases that train the next model. Caps per customer and per item, plus exploration of less popular items, would reduce this.
- **Explainability.** Every recommendation carries a reason (bought before, similar customers, similar style, trending) and, for collaborative picks, the purchased item that triggered it. Reasons come from the component that contributed most to the score, so they are faithful to how the list was built.

In [12]:
config.DOCS_DIR.mkdir(exist_ok=True)
pd.concat({"exposure": exposure_table, "age groups": fair, "variety": bubble, "variants": variant_check, "outfit completion": outfit_results}).to_csv(
    config.DOCS_DIR / "responsible_and_outfit_metrics.csv")
print("Saved docs/responsible_and_outfit_metrics.csv")

Saved docs/responsible_and_outfit_metrics.csv


## Findings to write up in your own words
1. **Outfit completion: does the basket model beat trending and content baselines, and by how much (with the bootstrap interval)?** ...
2. **What the examples show: are the suggestions plausible complements (different types, matching sets or seasons)?** ...
3. **Effect of collapsing colour variants on accuracy and variety:** ...
4. **Popularity bias and exposure: which model concentrates exposure, and what does that mean for sellers and customers?** ...
5. **Fairness across age groups, with the caveat about small groups:** ...
6. **Filter bubbles and the repurchase share, and why the prototype separates shelves:** ...
7. **Cold start: how large is the problem and what the content slice shows:** ...
8. **Privacy, manipulation and explainability: what you would require before real deployment:** ...